## Imports

In [16]:
%load_ext autoreload
%autoreload 2

import pickle
import pandas as pd
import numpy as np

import clip
import open_clip
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torchvision.models import convnext_base, ConvNeXt_Base_Weights, resnet50, ResNet50_Weights, vit_b_32, ViT_B_32_Weights
import matplotlib.pyplot as plt

from deepfake_utils.datasets import DeepFakeDataset
from deepfake_utils.train import train_epoch, validate_epoch, train
from deepfake_utils.models import MyModel

from PIL import Image

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

In [3]:
# model, _, preprocess = open_clip.create_model_and_transforms('convnext_base', pretrained='laion400m_s13b_b51k', device = device)

In [5]:
image_dir_path = 'Deepfake-Eval-2024/image-data'
model_type = 'ResNet-50-CLIP'

# small subset of training data to run on CPU and debug functional issues
debug_data = DeepFakeDataset("image-metadata-debug.csv", image_dir_path, model_type, is_train = True)
debug_data_loader = DataLoader(debug_data, batch_size = 32, shuffle = False)

In [7]:
model(X)

tensor([[-0.0040, -0.0087],
        [-0.0345, -0.0542],
        [-0.0403, -0.0259],
        [-0.0375, -0.0313],
        [-0.0488,  0.0417],
        [-0.0205,  0.0118],
        [ 0.0379,  0.0290],
        [ 0.0061,  0.0193],
        [-0.0015,  0.0002],
        [-0.0043, -0.0214],
        [-0.0408,  0.0326],
        [-0.0157,  0.0142],
        [-0.0311,  0.0290],
        [-0.0337, -0.0088],
        [ 0.0285, -0.0305],
        [ 0.0100,  0.0144],
        [-0.0327, -0.0057],
        [-0.0189, -0.0062],
        [-0.0111, -0.0157],
        [-0.0210, -0.0203],
        [-0.0217,  0.0077],
        [-0.0150,  0.0463],
        [ 0.0092,  0.0386],
        [-0.0088, -0.0079],
        [-0.0370, -0.0619],
        [-0.0286, -0.0416],
        [ 0.0317, -0.0127],
        [ 0.0147,  0.0201],
        [-0.0613, -0.0015],
        [-0.0609,  0.0013],
        [-0.0255, -0.0199],
        [-0.0257, -0.0056]], device='cuda:0', dtype=torch.float16,
       grad_fn=<AddmmBackward0>)

In [19]:
model_type = 'ResNet-50-CLIP'
train_data = DeepFakeDataset("image-metadata-train.csv", image_dir_path, model_type, is_train = True)
train_data_loader = DataLoader(train_data, batch_size = 32, shuffle = True)
val_data = DeepFakeDataset("image-metadata-val.csv", image_dir_path, model_type, is_train = False)
val_data_loader = DataLoader(val_data, batch_size = 32, shuffle = False)
model = MyModel("ResNet-50-pretrained-clip", device, 2)

In [23]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-3)
loss, roc_auc, pr_auc, acc = train(3, train_data_loader, val_data_loader, model, nn.CrossEntropyLoss(), optimizer, device, True, False, 1)

Epoch 1
- - - - - - - - - - - - - - - - - - - - - - - - - 
Training...
TEST loss: nan
TEST loss: nan
TEST loss: nan
TEST loss: nan
TEST loss: nan
TEST loss: nan
TEST loss: nan
TEST loss: nan
TEST loss: nan
TEST loss: nan
	Training Progress: 	[  320/ 1365]
TEST loss: nan


KeyboardInterrupt: 

In [15]:
loss, roc_auc, pr_auc, acc

(nan, 0.45264031538541344, 0.5685667991638184, 0.5950000286102295)